Let's solve a system of PDEs over $(t,x)\in[0,\infty)\times[-1,1]\times[S]$, where $S$ is a periodic n-dimensional space. Since we are solving problems on a finite computer, the spatial series representation of the system must be truncated to N terms, permitting the commutation of the sums and derivatives. However, to ensure accuracy, the infinite-dimensional linear algebra needs to be considered carefully.

To convert the Chebyshev polynomial $T_n(x)$ to the Ultraspherical polynomial we need two matrices: one to convert the $T_n(x)$ to the $C^1_n(x)$, which are equivalent to the Chebyshev polynomials of the second kind, and then the second conversion from $C^1_n(x)$ to $C^2_n(x)$.  The first matrix $S_0$, follows from Chebyshev recurrence relationships and is

$$
S^0 = \begin{bmatrix} 
1 & 0 & -1/2 & 0 &\cdots \\
0 & 1/2 & 0 & -1/2 & 0 &\cdots \\
0 & 0 & \ddots & \ddots & \ddots
\end{bmatrix}
$$

The second connection between derivative orders $\lambda$ as $C_n^\lambda(x)$ and $C_n^{\lambda+1}(x)$ can be found in Olver & Townsend 2013, with

$$
S^\lambda = \begin{bmatrix} 
1 & 0 & -\frac{\lambda}{\lambda+2} & 0 &\cdots \\
0 & \frac{\lambda}{\lambda+1} & 0 & -\frac{\lambda}{\lambda+3} & 0 &\cdots \\
0 & 0 & \frac{\lambda}{\lambda+2} & 0 & -\frac{\lambda}{\lambda+4} & 0 &\cdots \\
0 & 0 & \ddots & \ddots & \ddots
\end{bmatrix}
$$

Over the non-periodic dimension, the $\lambda$-order derivative matrix $D^\lambda$ has the first $\lambda$ columns as zeros and then the single super-diagonal pattern 

$$
D^\lambda =2^{\lambda-1}\Gamma(\lambda)\begin{bmatrix}
0&\cdots & 0& \lambda & 0 & \cdots \\
0&\cdots & 0& 0& \lambda+1 & 0 & \cdots \\
0&\cdots & 0& 0& 0& \lambda+2 & 0 & \cdots\\
\vdots& & \cdots & & & &\ddots & 0 & \cdots
\end{bmatrix}
$$

The next step is to approximate the time derivative. Let's use IMEX methods, which have two Butcher tableaux, one for the implicit terms and one for the explicit terms.  This generalization permits a wide range of time-stepping methods to be implemented.

We then need to incorporate boundary conditions.  This can be done in several ways... via taus (extra degrees of freedom) or via row deletion.  Olver and Townsend use row deletion. So, we'll go with that for now (plus it is closer to the current Rayleigh implemention). The coefficient condition will be implemented on the left-hand side of the equation and the boundary values will replace some rows on the right-hand side. For instance, letting $B_2$ be the boundary operator, $P_{N-2}$ a restriction operator that removes the last two rows of $A_{-}$ and $A_{+}$, and $\mathbf{c}$ be the boundary values, the matrix equations become

$$
\begin{bmatrix}
B_2\\
P_{N-2}A_{-}
\end{bmatrix}\mathbf{q}^{k+1} = 
\begin{bmatrix}
\mathbf{0}\\
P_{N-2}A_{+}
\end{bmatrix}\mathbf{q}^k + \mathbf{c}.
$$

Therefore, after inverting the left-hand side matrix via some factorization, at each stage of the IMEX method, one can step forward in time. In spectral space, the multiplication matrices link two scalar functions $a(x)$ and $u(x)$ to the same Ultraspherical bases, with

$$a(x)u(x) = \sum_{j=0}^M\sum_{k=0}^N\sum_{s=\mathrm{max}(0,k-j)}^k a_{2 s + j - k} c_s^\lambda(k,2 s + j -k)u_k C^\lambda_s(x),$$

where the $c_s^\lambda$ are the connection coefficients (see Olver & Townsend 2013 for details).  That is the multiplication matrix $M^\lambda[a]$ has the elements

$$M^\lambda[a]_{j,k} = \sum_{s=\mathrm{max}(0,k-j)}^k a_{2 s + j - k} c_s^\lambda(k,2 s + j -k).$$

Note here $a(x)$ is expressed on the $C^\lambda(x)$ basis and so if one defines this scalar over the Chebyshev polynomials of the first kind, then one must create additional conversion matrices to bring this field onto the appropriate Ultraspherical basis.  Note that this actually permits us to take the Frechet derivative to compute the Jacobian necessary to implicitly step nonlinear terms as well using Newton's method.

A library in Python, C, and Fortran using Ultraspherical polynomials to build sparse spectral methods for PDEs (ULTRA). Copyright (C) 2023 Kyle Augustson.

In [ ]:
# Include necessary packages for sparse linear algebra and plotting.
import numpy as np
import scipy as sp
import scipy.sparse as sparse   
    
# This is the forward Chebyshev T_n (Chebyshev polynomials of the first kind) transform
# to coefficient space
def dct_chebyshev_transform(f):
    N = len(f)
    coefs = f
    coefs = sp.fft.dct(coefs,orthogonalize=True,norm='ortho')/np.sqrt(0.5*N)
    coefs[0] = coefs[0]/np.sqrt(2)
    return coefs

# This is the backward Chebyshev T_n transform (to physical space)
def idct_chebyshev_transform(f):
    N = len(f)
    icoefs = np.sqrt(N/2)*f
    icoefs[0] = np.sqrt(2)*icoefs[0]
    icoefs = sp.fft.idct(icoefs,orthogonalize=True,norm='ortho')
    return icoefs

# Create the Chebshev collocation points for the T_n
def get_collocation_points(x0,x1,N):
    #Map to Chebyshev domain
    x = np.pi*(np.arange(N)+0.5)/np.float64(N)
    x = 0.5*(x1+x0)-0.5*(x1-x0)*np.cos(x)
    return x

# Create a connection coefficient when multiplying two fields 
# expressed over the Ultraspherical polynomials C^\lambda
def multiplication_coef(s,j,k,lam):
    t1 = j + k + lam
    t2 = t1 - s
    t1 = t1 - 2*s
    
    #First term
    cs = t1/t2
    
    t2 = 1
    for t in range(s - 1):
        t2 = t2*(lam + t)/(1 + t)
    
    #Second term
    cs = cs*t2
    
    t3 = 1 
    for t in range(j - s - 1):
        t3 = t3*(lam + t)/(1 + t)
    
    #Third term
    cs = cs*t3
    
    t4 = 1
    for t in range(s - 1):
        t1 = j + k - 2*s + t
        t2 = t1 + lam
        t1 = t1 + 2*lam
        t1 = t1/t2
        t4 = t4*t1
    
    #Fourth term
    cs = cs*t4
    
    t5 = 1
    for t in range(j-s-1):
        t1 = k - s + t
        t2 = t1 + lam
        t1 = t1 + 1
        t1 = t1/t2
        t5 = t5*t1
        
    #Fifth term
    cs = cs*t5
    return cs

# Given a non-constant coefficient that has been expanded over the C^\lambda,
# giving the set of coefficients (coefs), construct the multiplication matrix
# of size M x N and order \lambda (lam)
def build_multiplication_matrix(coefs,M,N,lam):
    Mlam = sparse.lil_matrix((M,N),dtype=np.float64)
    ncoef = len(coefs)
    for j in range(M):
        for k in range(N):
            t = 0
            i1 = int(np.max([0,k-j]))
            inds = np.array(2*np.arange(i1,k,1)+j-k).astype(int)
            ainds = np.where((inds>=0)&(inds<ncoef))[0]
            sinds = np.arange(i1,k,1,dtype=int)
            for a in ainds:
                cs = multiplication_coef(sinds[a],k,inds[a],lam)
                t = t + cs*coefs[inds[a]]
            if (t!=0): 
                Mlam[j,k] = t
    Mlam.tocsc()
    return Mlam

# Construct the derivative matrix of order \lambda (lam) and size M x N
def build_derivative_matrix(M,N,lam):
    elements = 2**(lam-1)*np.math.factorial(lam-1)*(np.arange(N,dtype=np.float64)+lam)
    Dlam = sparse.diags(elements,lam,shape=(M,N),format='csc')
    return Dlam

# Construct the matrices that convert coefficients expressed over C^\lambda to C^{\lambda +1}
# S0 converts coefficients of a field a(x) from T_n  to C^1
# S(\lambda) converts coefficients of a field a(x) from C^\lambda to C^{\lambda +1}
def build_conversion_matrix(M,N,lam):
    if (lam==0):
        upper = -np.ones(N-2)*0.5
        diag = np.ones(N)*0.5
        diag[0] = 1
        Slam = sparse.diags([diag,upper],[0,2],shape=(M,N),format='csc')
    else:
        ran = np.arange(N-2)+lam+2
        upper = -lam*np.ones(N-2)/ran
        ran = np.arange(N)+lam
        diag = lam*np.ones(N)/ran
        Slam = sparse.diags([diag,upper],[0,2],shape=(M,N),format='csc')   
    return Slam

# This constructs the sparse representation of Dirichlet and Neumann (and other) 
# boundary conditions as size nvar*N x nvar*N. Recall these act directly on the
# coefficients projected on the T^n.
def build_boundary_matrix(N)
    Blam = sparse.lil_matrix((nvar*N,nvar*N),dtype=np.float64)
    # Lower boundary
    boundary_row = 0
    for var in range(nvar):
        if (boundaries[var][0] is not None):
            nbd = len(boundaries[var])
            for bd in range(nbd):
                boundary_type = boundaries[var][bd]
                
                if (boundary_type==0): # Integral condition = 0 (for gauges)
                    coefs = np.zeros(N,dtype=np.float64)
                    for k in range(0,N,2):
                        coefs[k] = 2/(1-k*k)
                elif (boundary_type==-1): # Left Dirichlet = -1
                    coefs = np.zeros(N,dtype=np.float64)
                    for k in range(N):
                        coefs[k] = 1 - 2*(k % 2)
                elif (boundary_type==-2): # Left Neumann = -2
                    coefs = np.zeros(N,dtype=np.float64)
                    for k in range(1,N,1):
                        coefs[k] = -(1 - 2*(k % 2))*k*k
                elif (boundary_type<-2): # Left higher-order = -p (p = order+1)
                    coefs = np.zeros(N,dtype=np.float64)
                    p = -boundary_type-1
                    for k in range(p,N,1):
                        tmp = 1
                        for j in range(p):
                            tmp = tmp*((k*k-j*j)/(2*j+1))
                        coefs[k] = (1 - 2*(k+p % 2))*tmp
                elif (boundary_type==1): # Right Dirichlet = 1
                    coefs = np.ones(N,dtype=np.float64)
                elif (boundary_type=2): # Right Neumann = 2
                    coefs = np.arange(N,dtype=np.float64)**2
                elif (boundary_type>2): # Right higher-order = p (p = order+1)
                    coefs = np.zeros(N,dtype=np.float64)
                    p = boundary_type-1
                    for k in range(p,N,1):
                        tmp = 1
                        for j in range(p):
                            tmp = tmp*((k*k-j*j)/(2*j+1))
                        coefs[k] = tmp
                else:
                    print('Boundary type error for variable '+var+' for boundary '+bd)
                Blam[boundary_row,var*N:(var+1)*N] = coefs
                boundary_row = boundary_row + 1
    return Blam.tocsc()

# Build the problem matrix given a set of nccs
# nccs -- a structure of nccs of size nvar x nvar x nderiv (number of equations equals number of variables)
# N -- Initial matrix size (adaptive QR solver will change this, but allows error monitoring)
def build_problem_matrices(nccs,N):
    B = build_boundary_matrix(N)
    L = build_linear_matrix(nccs[:,,:,1:],N)
    M = build_mass_matrix(nccs[:,0,0],N)
    return (B,L,M)

def build_mass_matrix(ncc,N):
    #Convert to C^(\lambda_max), with lambda_max = max(lambda over eqns)
    mass = sparse.lil_matrix((nvar*N,nvar*N),dtype=np.float64)
    Ncur = nboundaries
    for var in range(nvar):
        cur_ncc = ncc[var]
        if ncc[var] is not None:
            if (boundaries[var][0] is not None):
                nbd = len(boundaries[var])
            else:
                nbd = 0
            Nnext = Ncur+N-nbd
            m0 = len(cur_ncc)
            if (m0>1):
                tmp = build_multiplication_matrix(cur_ncc,N+m0,N,lam)
                tmp = (build_conversion_matrix(N+2,N+m0,0))@tmp
            else:
                tmp = cur_ncc[0]
                tmp = (build_conversion_matrix(N+2,N,0))*tmp
            
            for i in range(1,nderiv-1,1):
                tmp = (build_conversion_matrix(N+2,N+2,i))@tmp
            tmp = (build_conversion_matrix(N-nbd,N+2,nderiv-1))@tmp
            mass[Ncur:Nnext,var*N:(var+1)*N] = tmp
        else:
            Nnext = Ncur+N
        Ncur = Nnext
    return mass

def build_linear_matrix(nccs,N):
    #Convert to C^(\lambda_max), with lambda_max = max(lambda over eqns)
    linear = sparse.lil_matrix((nvar*N,nvar*N),dtype=np.float64)
    Ncur = nboundaries
    for eq in range(nvar):
        if (boundaries[eq][0] is not None):
            nbd = len(boundaries[eq])
        else:
            nbd = 0
        Nnext = Ncur + N - nbd
        for var in range(nvar):
            deriv = 0
            cur_ncc = nccs[eq,var,deriv]
            if cur_ncc is not None:
                ml = len(cur_ncc)
                if (ml>1):
                    tmp = build_multiplication_matrix(cur_ncc,N+ml,N,deriv)
                    tmp = (build_conversion_matrix(N+2,N+ml,0))@tmp
                else:
                    tmp = cur_ncc[0]
                    tmp = (build_conversion_matrix(N+2,N,0))*tmp
                for i in range(deriv+1,nderiv-1,1):
                    tmp = (build_conversion_matrix(N+2,N+2,i))@tmp
                tmp = (build_conversion_matrix(N-nbd,N+2,nderiv-1))@tmp
                linear[Ncur:Nnext,var*N:(var+1)*N] += tmp
            for deriv in range(1,nderiv,1):
                cur_ncc = nccs[eq,var,deriv]
                if cur_ncc is not None:
                    ml = len(cur_ncc)
                    tmp = build_derivative_matrix(N+deriv,N,deriv)
                    if (ml>1):
                        tmp = (build_multiplication_matrix(cur_ncc,N+ml,N+deriv,deriv))@tmp
                        tmp = (build_conversion_matrix(N+2,N+ml,deriv))@tmp
                    else:
                        tmp1 = cur_ncc[0]
                        tmp = tmp1*(build_conversion_matrix(N+2,N+deriv,deriv))@tmp
                    for i in range(deriv+1,nderiv-1,1):
                        tmp = (build_conversion_matrix(N+2,N+2,i))@tmp
                    tmp = (build_conversion_matrix(N-nbd,N+2,nderiv-1))@tmp
                    linear[Ncur:Nnext,var*N:(var+1)*N] += tmp
            deriv = nderiv
            cur_ncc = nccs[eq,var,deriv]
            if cur_ncc is not None:
                ml = len(cur_ncc)
                if (ml>1):
                    tmp = build_derivative_matrix(N+ml,N,deriv)
                    tmp = build_multiplication_matrix(cur_ncc,N-nbd,N+ml,deriv)
                else:
                    tmp = cur_ncc[0]
                    tmp = build_derivative_matrix(N-nbd,N,deriv)*tmp
                linear[Ncur:Nnext,var*N:(var+1)*N] += tmp
        Ncur = Nnext
    return linear


class BVP_Solver:
    def __init__(self):
        self.B = 0
        self.L = 0
        self.M = 0
        self.nccs = 0
        self.N = 0
        self.A = 0

class EVP_Solver:
    def __init__(self):
        self.B = 0
        self.L = 0
        self.M = 0
        self.nccs = 0
        self.N = 0
        self.A = 0

# n-stage IMEX Runge-Kutta, IVP
# First row of nccs contains the weights for the assumed only first order time derivatives
# That is, each equation only contains at most a single time derivative of the corresponding variable
# Can be updated in the future for more complex systems.
# This also means that if your equation contains a higher-order time derivative, then 
# it needs to be reduced to first order.
class IVP_Solver:
    def __init__(self):
        self.B = None
        self.L = None
        self.M = None
        self.nccs = None
        self.N = None
        self.A = None
        self.butcher = None
        self.dt = None
        self.forcing = None
        self.forcing_function = None
        self.state = None
        self.explicit = None
        self.implicit = None
        self.time = None
        self.nonlinear = None
    
    def set_dt(self,dt)
        self.dt = dt
    
    def set_butcher(self,butcher):
        self.butcher = butcher

    def set_nccs(self,nccs):
        self.nccs = nccs
        
    def set_N(self,N):
        self.N = N
    
    def set_state(self,state):
        self.state = state
    
    def build_solver(self):
        if (self.B is None):
            self.B, self.L, self.M = build_problem_matrices(self.nccs,self.N)
        stages = len(self.butcher[:,1])
        self.A = ()
        for i in range(stages):
            tmp = self.M + (self.dt*self.butcher[i,i,1])*self.L + self.B
            self.A.append(sparse.splu(tmp))
            
    def update_dt(self,dt):
        set_dt(self,dt)
        build_solver(self)
    
    def set_forcing(self,s,func):
        self.forcing = func(self,self.time+self.dt*butcher[s,0,0])

    def compute_nonlinear(self):
        nl = self.nonlinear_function(self,new_state)
        return nl
        
    def timestep(self):
        stages = len(butcher[:,0])
        for i in range(stages):
            self.set_forcing(self,i,self.forcing_function)
            um = self.state - self.dt*(np.sum(self.butcher[i,1:,0]*self.explicit + self.butcher[i,1:,1]*self.implicit))
            up = self.A[i].solve(self.M.dot(um+self.forcing)+boundary_values)
            self.explicit[i] = self.compute_nonlinear(self,up)
            self.implicit[i] = (up-um)/(self.butcher[i,i+1,1]*self.dt)
        self.state  = self.state - self.dt*(np.sum(self.butcher[nbutcher,1:,0]*self.explicit + self.butcher[nbutcher,1:,1]*self.implicit))